# 🎓 E.S.I.S (Electronic Smoking Identification System)
## Modul AI Face Recognition: YuNet + FaceNet
**Metodologi:** CRISP-DM (*Cross-Industry Standard Process for Data Mining*)  
**Tujuan:** Mengidentifikasi identitas mahasiswa/pelaku yang tertangkap kamera CCTV saat aktivitas merokok terdeteksi.

---
### 📌 Ringkasan Arsitektur Model:
1. **Face Detector: YuNet (OpenCV)**  
   - Menggantikan MTCNN lama.
   - Sangat cepat (~5-15 ms per frame), ringan, native C++/OpenCV, tanpa *memory leak*, dan siap deploy ke TensorRT / ONNX Runtime.
2. **Face Feature Extractor: FaceNet (Inception-ResNet v1)**  
   - Menghasilkan vektor embedding 512-dimensi yang unik untuk setiap wajah.
3. **Face Classifier & Matching:**  
   - Menggunakan perpaduan **Euclidean Distance / Cosine Similarity thresholding** dan **Linear SVM Classifier** berprobabilitas.
4. **Target Deployment:**  
   - **Local PC / Laptop (Nvidia MX550)** atau **Cloud GPU (RunPod Nvidia RTX A6000)** menggunakan **TensorRT / PyTorch CUDA**.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT & DEPENDENCIES
# ==============================================================================
# Instalasi library yang dibutuhkan
!pip install -q keras-facenet opencv-python scikit-learn seaborn matplotlib tqdm

import os
import sys
import shutil
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as Img
from numpy import asarray, expand_dims
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from keras_facenet import FaceNet
import pickle

print("✅ Dependency berhasil dimuat!")

# ==============================================================================
# 2. DOWNLOAD PRETRAINED YUNET MODEL (OpenCV Zoo)
# ==============================================================================
yunet_model_path = "face_detection_yunet_2023mar.onnx"
yunet_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"

if not os.path.exists(yunet_model_path):
    print("⬇️ Sedang mengunduh model YuNet (OpenCV Zoo)...")
    urllib.request.urlretrieve(yunet_url, yunet_model_path)
    print("✅ Model YuNet berhasil diunduh:", yunet_model_path)
else:
    print("✅ Model YuNet sudah tersedia:", yunet_model_path)


---
## 📊 2. DATA UNDERSTANDING
Memuat dataset foto wajah mahasiswa volunteer dari Google Drive dan memvisualisasikan distribusi jumlah foto per kelas.


In [ ]:
# ==============================================================================
# 3. MOUNT GOOGLE DRIVE & EKSPLORASI DATASET
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ SESUAIKAN DENGAN PATH DATASET WAJAH DI GOOGLE DRIVE ANDA
dataset_raw_path = '/content/drive/MyDrive/DATASET/WAJAH/'

if not os.path.exists(dataset_raw_path):
    print(f"⚠️ Folder '{dataset_raw_path}' tidak ditemukan!")
    print("Silakan periksa path folder dataset wajah Anda di Google Drive.")
else:
    class_names = []
    image_counts = []

    for folder in sorted(os.listdir(dataset_raw_path)):
        folder_path = os.path.join(dataset_raw_path, folder)
        if os.path.isdir(folder_path) and not folder.startswith('.'):
            valid_images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(valid_images) > 0:
                class_names.append(folder)
                image_counts.append(len(valid_images))

    print(f"Total Kelas Terdaftar: {len(class_names)} orang/mahasiswa")
    print(f"Total Gambar: {sum(image_counts)} foto")

    # Visualisasi Distribusi Dataset
    plt.figure(figsize=(15, 6))
    ax = sns.barplot(x=class_names, y=image_counts, hue=class_names, palette='viridis', legend=False)
    plt.xlabel('Nama Mahasiswa / Kelas', fontsize=12, fontweight='bold')
    plt.ylabel('Jumlah Foto', fontsize=12, fontweight='bold')
    plt.title('Distribusi Jumlah Foto Dataset Wajah per Kelas', fontsize=15, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, max(image_counts) + 5)

    for container in ax.containers:
        ax.bar_label(container, padding=3, fmt='%d')

    plt.tight_layout()
    plt.show()


---
## 🧹 3. DATA PREPARATION (Face Detection & Cropping dengan YuNet)
Pada tahap ini, kita mendeteksi wajah pada foto mentah menggunakan **YuNet**, memotong (*crop*) area wajah dengan *margin padding* proporsional, lalu mengubah ukurannya menjadi **160x160 piksel** (standar input FaceNet).


In [ ]:
# ==============================================================================
# 4. KELAS DETEKTOR WAJAH YUNET
# ==============================================================================
class YuNetFaceDetector:
    """Wrapper untuk YuNet Face Detector OpenCV"""
    def __init__(self, model_path="face_detection_yunet_2023mar.onnx", conf_threshold=0.85, nms_threshold=0.3):
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
        self.model_path = model_path
        # Inisialisasi detector dengan ukuran default (akan diupdate per frame)
        self.detector = cv2.FaceDetectorYN.create(
            model=model_path,
            config="",
            input_size=(320, 320),
            score_threshold=conf_threshold,
            nms_threshold=nms_threshold,
            top_k=5000
        )

    def detect_face(self, img_bgr, margin_percent=0.15):
        """
        Mendeteksi wajah dan mengembalikan crop wajah RGB (160x160) serta koordinat bounding box (x, y, w, h).
        Jika tidak ada wajah yang memenuhi threshold, mengembalikan (None, None).
        """
        h, w, _ = img_bgr.shape
        self.detector.setInputSize((w, h))
        _, faces = self.detector.detect(img_bgr)

        if faces is None or len(faces) == 0:
            return None, None

        # Ambil wajah dengan confidence tertinggi
        best_face = max(faces, key=lambda f: f[-1])
        conf = best_face[-1]

        if conf < self.conf_threshold:
            return None, None

        # Bounding box YuNet: [x, y, w, h]
        x, y, bw, bh = map(int, best_face[:4])

        # Berikan margin ekstra agar dagu, dahi, dan telinga tidak terpotong ketat
        pad_x = int(bw * margin_percent)
        pad_y = int(bh * margin_percent)

        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(w, x + bw + pad_x)
        y2 = min(h, y + bh + pad_y)

        # Potong dan ubah ke RGB
        face_crop = img_bgr[y1:y2, x1:x2]
        if face_crop.size == 0:
            return None, None

        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        face_pil = Img.fromarray(face_rgb).resize((160, 160))

        return np.asarray(face_pil), (x1, y1, x2 - x1, y2 - y1)

# Inisialisasi Detektor YuNet
yunet = YuNetFaceDetector(model_path=yunet_model_path, conf_threshold=0.85)
print("✅ Detektor Wajah YuNet siap digunakan!")


In [ ]:
# ==============================================================================
# 5. PREPROCESSING SELURUH DATASET KE FOLDER BERSIH (160x160)
# ==============================================================================
output_preprocessed_dir = '/content/DATA_PREPROCESSED'

if os.path.exists(output_preprocessed_dir):
    shutil.rmtree(output_preprocessed_dir)
os.makedirs(output_preprocessed_dir)

sample_visualizations = []
success_count = 0
fail_count = 0

print(f"🚀 Memulai Preprocessing dataset dari '{dataset_raw_path}'...")

for class_name in tqdm(class_names, desc="Memproses Kelas"):
    class_src_dir = os.path.join(dataset_raw_path, class_name)
    class_dst_dir = os.path.join(output_preprocessed_dir, class_name)
    os.makedirs(class_dst_dir, exist_ok=True)

    has_sample = False

    for fname in os.listdir(class_src_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        file_path = os.path.join(class_src_dir, fname)
        img = cv2.imread(file_path)
        if img is None:
            continue

        face_160, bbox = yunet.detect_face(img)

        if face_160 is not None:
            save_path = os.path.join(class_dst_dir, fname)
            Img.fromarray(face_160).save(save_path)
            success_count += 1

            if not has_sample:
                sample_visualizations.append((face_160, class_name))
                has_sample = True
        else:
            fail_count += 1

print(f"\n✅ Preprocessing Selesai!")
print(f"   - Wajah berhasil dipotong (160x160): {success_count} foto")
print(f"   - Foto gagal/tidak ada wajah terdeteksi: {fail_count} foto")

# Visualisasi Sampel Hasil Preprocessing YuNet
if sample_visualizations:
    cols = 5
    rows = (len(sample_visualizations) + cols - 1) // cols
    plt.figure(figsize=(15, 3 * rows))
    for i, (sample_img, label) in enumerate(sample_visualizations):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(sample_img)
        plt.title(label[:15], fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


---
## 🧠 4. MODELLING (Feature Extraction dengan FaceNet & Training Classifier)
Setiap wajah yang telah dipotong berukuran 160x160 piksel diubah menjadi vektor representasi numerik (embedding) **512 dimensi** menggunakan **FaceNet (Inception-ResNet v1)**.
Setelah itu, kita membangun:
1. **Basis Data Embedding Wajah (.npz)** untuk pencocokan jarak Euclidean / Cosine.
2. **Model Klasifikasi Linear SVM (.pkl)** untuk inferensi cepat dengan probabilitas klasifikasi.


In [ ]:
# ==============================================================================
# 6. FEATURE EXTRACTION MENGGUNAKAN FACENET
# ==============================================================================
print("⏳ Memuat model FaceNet (Inception-ResNet-v1)... ")
facenet = FaceNet()
print("✅ FaceNet berhasil dimuat!")

X_embeddings = []
Y_labels = []

# Iterasi folder data bersih hasil preprocessing
for class_name in sorted(os.listdir(output_preprocessed_dir)):
    class_dir = os.path.join(output_preprocessed_dir, class_name)
    if not os.path.isdir(class_dir):
        continue

    files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(files) == 0:
        continue

    # Ekstraksi batch per kelas
    images_batch = []
    for f in files:
        img_path = os.path.join(class_dir, f)
        img = Img.open(img_path).convert('RGB')
        images_batch.append(asarray(img))

    # Ekstrak embedding 512 dimensi
    embeddings = facenet.embeddings(images_batch)

    for emb in embeddings:
        X_embeddings.append(emb)
        Y_labels.append(class_name)

X_embeddings = np.array(X_embeddings)
Y_labels = np.array(Y_labels)

print(f"\n✅ Total Embedding Terbentuk: {X_embeddings.shape[0]} vektor wajah")
print(f"   Dimensi per Vektor Wajah: {X_embeddings.shape[1]} fitur")

# Simpan Database Embedding ke file .npz
npz_output_path = "data_wajah_facenet.npz"
np.savez_compressed(npz_output_path, x=X_embeddings, y=Y_labels)
print(f"💾 File database embedding berhasil disimpan: '{npz_output_path}'")


In [ ]:
# ==============================================================================
# 7. TRAINING CLASSIFIER (LINEAR SVM DENGAN PROBABILITY)
# ==============================================================================
# Bagi data menjadi Train (80%) dan Test (20%) untuk evaluasi objektif
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, Y_labels, test_size=0.20, random_state=42, stratify=Y_labels
)

print(f"Data Latih: {len(y_train)} sampel | Data Uji: {len(y_test)} sampel")

# Latih Linear SVM dengan probability=True agar menghasilkan nilai keyakinan (confidence)
svm_model = SVC(kernel='linear', probability=True, C=1.0)
svm_model.fit(X_train, y_train)

# Simpan model SVM terlatih
svm_output_path = "face_classifier_svm.pkl"
with open(svm_output_path, "wb") as f:
    pickle.dump(svm_model, f)

print(f"✅ Model SVM berhasil dilatih dan disimpan: '{svm_output_path}'")


---
## 📈 5. EVALUATION
Pengujian performa model menggunakan metrik standar Computer Vision:
- **Accuracy Score**
- **Precision, Recall, F1-Score** (Classification Report)
- **Confusion Matrix**


In [ ]:
# ==============================================================================
# 8. EVALUASI MODEL PADA DATA UJI (TEST SET)
# ==============================================================================
y_pred = svm_model.predict(X_test)
y_proba = svm_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
print("=" * 60)
print(f"🎯 AKURASI MODEL FACE RECOGNITION (TEST SET): {acc * 100:.2f}%")
print("=" * 60)

print("\n📋 Laporan Klasifikasi Detail:")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=svm_model.classes_)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=svm_model.classes_,
            yticklabels=svm_model.classes_)
plt.title(f'Confusion Matrix Face Recognition (YuNet + FaceNet + SVM)\nAkurasi: {acc*100:.2f}%', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi AI', fontsize=12)
plt.ylabel('Label Asli (Ground Truth)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


---
## 📷 6. PENGUJIAN REAL-TIME DENGAN WEBCAM (GOOGLE COLAB)
Uji coba sistem pengenalan wajah langsung dari kamera laptop Anda menggunakan tombol *Capture*.


In [ ]:
# ==============================================================================
# 9. FUNGSI CAPTURE WEBCAM DI GOOGLE COLAB
# ==============================================================================
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capture_colab_photo(filename='webcam_test.jpg', quality=0.85):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Ambil Foto Wajah';
      capture.style.padding = '10px 20px';
      capture.style.fontSize = '16px';
      capture.style.backgroundColor = '#4CAF50';
      capture.style.color = 'white';
      capture.style.border = 'none';
      capture.style.borderRadius = '8px';
      capture.style.cursor = 'pointer';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.marginTop = '10px';
      video.style.borderRadius = '8px';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# ==============================================================================
# 10. EKSEKUSI PENGUJIAN IDENTITAS DARI FOTO WEBCAM
# ==============================================================================
try:
    print("Silakan klik tombol '📸 Ambil Foto Wajah' yang muncul di bawah:")
    test_photo_file = capture_colab_photo()
    print("✅ Foto berhasil diambil. Menganalisis wajah...")

    frame = cv2.imread(test_photo_file)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Deteksi wajah dengan YuNet
    face_160, bbox = yunet.detect_face(frame)

    if face_160 is not None:
        x, y, w, h = bbox

        # Ekstrak Embedding FaceNet
        emb = facenet.embeddings(np.expand_dims(face_160, axis=0))

        # 1. Prediksi via SVM Probability
        probas = svm_model.predict_proba(emb)[0]
        max_idx = np.argmax(probas)
        confidence = probas[max_idx]
        svm_pred_label = svm_model.classes_[max_idx]

        # 2. Verifikasi Euclidean Distance terhadap Database Wajah
        distances = [np.linalg.norm(emb[0] - db_emb) for db_emb in X_embeddings]
        min_dist = min(distances)
        dist_match_label = Y_labels[np.argmin(distances)]

        # Ambang batas pengenalan (Threshold Jarak Euclidean & Confidence)
        EUCLIDEAN_THRESHOLD = 0.75
        CONFIDENCE_THRESHOLD = 0.60

        if min_dist < EUCLIDEAN_THRESHOLD and confidence >= CONFIDENCE_THRESHOLD:
            identified_name = svm_pred_label
            status_text = f"DIKENAL: {identified_name}"
            box_color = (0, 255, 0) # Hijau
        else:
            identified_name = "Unknown / Tidak Dikenal"
            status_text = "UNKNOWN (Bukan Volunteer)"
            box_color = (255, 0, 0) # Merah

        # Gambar Bounding Box dan Teks
        plt.figure(figsize=(8, 6))
        plt.imshow(frame_rgb)
        ax = plt.gca()
        rect = plt.Rectangle((x, y), w, h, fill=False, color='lime' if box_color==(0,255,0) else 'red', linewidth=3)
        ax.add_patch(rect)

        caption = f"{identified_name}\nConf: {confidence*100:.1f}% | Dist: {min_dist:.2f}"
        plt.text(x, max(20, y - 10), caption, color='white', fontsize=11, fontweight='bold',
                 bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.5'))

        plt.axis('off')
        plt.title("Hasil Pengenalan Wajah (YuNet + FaceNet)", fontsize=14, fontweight='bold')
        plt.show()

        print(f"\n🔎 Ringkasan Hasil:")
        print(f"   - Nama Teridentifikasi : {identified_name}")
        print(f"   - Confidence Score     : {confidence * 100:.2f}%")
        print(f"   - Jarak Euclidean      : {min_dist:.4f} (Batas threshold: {EUCLIDEAN_THRESHOLD})")
    else:
        print("⚠️ Tidak ada wajah yang terdeteksi pada foto webcam. Coba ambil foto dengan pencahayaan yang lebih terang.")

except Exception as e:
    print("Info:", e)


---
## 🚀 7. PANDUAN DEPLOYMENT: RUNPOD & TENSORRT VS OPENVINO

### ❓ Mengapa TensorRT, BUKAN OpenVINO untuk RunPod / Nvidia GPU?
1. **OpenVINO (Open Visual Inference & Neural Network Optimization):**
   - Dikembangkan oleh **Intel**, dioptimalkan secara spesifik untuk arsitektur prosesor Intel (Intel Core, Xeon, Intel Integrated GPU, dan Neural Compute Stick).
   - OpenVINO **TIDAK BISA** memanfaatkan kemampuan penuh GPU Nvidia CUDA atau Tensor Cores di RunPod.
2. **TensorRT:**
   - Dikembangkan oleh **Nvidia** khusus untuk ekosistem GPU Nvidia (termasuk **RTX A6000** di RunPod dan **GeForce MX550/RTX** di laptop).
   - TensorRT mengoptimasi graf model melalui *layer fusion*, *precision quantization* (FP16 & INT8), dan alokasi memori *kernel-level* sehingga inferensi berjalan **2x s.d. 5x lebih cepat** dibandingkan PyTorch murni.

---

### 📦 Berkas yang Wajib Diunduh dari Colab untuk AI Server:
Jalankan cell di bawah untuk mengunduh semua berkas model siap deploy:
1. `face_detection_yunet_2023mar.onnx` (Model Detektor Wajah YuNet)
2. `data_wajah_facenet.npz` (Vektor Embedding Database Wajah Mahasiswa)
3. `face_classifier_svm.pkl` (Model Klasifikasi SVM Berprobabilitas)


In [ ]:
# ==============================================================================
# 11. DOWNLOAD BERKAS MODEL UNTUK DEPLOYMENT KE AI SERVER
# ==============================================================================
from google.colab import files

print("⬇️ Menyiapkan unduhan berkas deployment...")
files_to_download = [
    "face_detection_yunet_2023mar.onnx",
    "data_wajah_facenet.npz",
    "face_classifier_svm.pkl"
]

for f in files_to_download:
    if os.path.exists(f):
        print(f"Mengunduh: {f}")
        try:
            files.download(f)
        except Exception as e:
            print(f"Gagal auto-download {f}: {e}")
    else:
        print(f"⚠️ Berkas {f} belum tersedia.")
